# 🚨 AWS Cost Anomaly Detection — Isolation Forest (Fixed)
**Multi-account ready** | Account-normalized features | No label encoding issues

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

print('✅ Imports OK')

## 2. Load Data

In [ ]:
# --- Option A: Upload directly ---
# from google.colab import files
# uploaded = files.upload()
# FILE_PATH = list(uploaded.keys())[0]

# --- Option B: Mount Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# FILE_PATH = '/content/drive/MyDrive/aws_billing_usage_usman.csv'

FILE_PATH = 'aws_billing_usage_usman.csv'

df = pd.read_csv(FILE_PATH, low_memory=False)
df['account_id'] = df['account_id'].astype(str)
print(f'Loaded: {df.shape[0]:,} rows | {df["account_id"].nunique()} account(s)')
df.head(2)

## 3. Feature Engineering

### Key design decisions:
- **Negative costs** (credits/refunds) are flagged separately and excluded from anomaly training — they are not cost anomalies
- **Frequency encoding** replaces label encoding — no more 'Other' bucket getting 100% anomaly rate
- **Account-normalized features**: all cost features are computed *relative to that account+service baseline*, not absolute values. This means a $100 spike for an account that normally spends $10 is treated the same as a $1000 spike for an account that normally spends $1000
- **RobustScaler** instead of StandardScaler — uses median/IQR, not mean/std, so extreme outliers don't distort the scaling

In [ ]:
# ── Parse timestamps ──────────────────────────────────────────────────────────
df['timestamp']   = pd.to_datetime(df['timestamp'],   utc=True, errors='coerce')
df['usage_start'] = pd.to_datetime(df['usage_start'], utc=True, errors='coerce')
df = df.sort_values('timestamp').reset_index(drop=True)

# ── Tag and remove credits/refunds — they are not cost anomalies ──────────────
df['is_credit'] = (df['cost'] < 0).astype(int)
df_train = df[df['cost'] >= 0].copy()
print(f'Credits/refunds excluded from training: {df["is_credit"].sum():,} rows')
print(f'Rows for training: {len(df_train):,}')

# ── Temporal features ─────────────────────────────────────────────────────────
df_train['hour']       = df_train['timestamp'].dt.hour
df_train['dayofweek']  = df_train['timestamp'].dt.dayofweek
df_train['day']        = df_train['timestamp'].dt.day
df_train['month']      = df_train['timestamp'].dt.month
df_train['is_weekend'] = (df_train['dayofweek'] >= 5).astype(int)

# ── Frequency encoding (replaces label encoding) ─────────────────────────────
# Value = how common is this service/usage_type (fraction of total rows)
# Common services get high values, rare ones get low values
# No arbitrary 'Other' bucket — every service gets its true frequency
svc_freq  = df_train['service'].value_counts(normalize=True).to_dict()
ut_freq   = df_train['usage_type'].value_counts(normalize=True).to_dict()
df_train['service_freq']    = df_train['service'].map(svc_freq).fillna(0)
df_train['usage_type_freq'] = df_train['usage_type'].map(ut_freq).fillna(0)

# ── ACCOUNT-NORMALIZED cost features ─────────────────────────────────────────
# Group key: account + service + usage_type
# This means: "is this row unusual FOR THIS ACCOUNT'S usage of THIS service?"
# A customer spending $10/day on EC2 and a customer spending $1000/day on EC2
# both get judged against THEIR OWN baseline, not a global average.
grp_key = ['account_id', 'service', 'usage_type']

df_train['acct_svc_mean'] = df_train.groupby(grp_key)['cost'].transform('mean')
df_train['acct_svc_std']  = df_train.groupby(grp_key)['cost'].transform('std').fillna(0)
df_train['acct_svc_p95']  = df_train.groupby(grp_key)['cost'].transform(lambda x: x.quantile(0.95))
df_train['acct_svc_median'] = df_train.groupby(grp_key)['cost'].transform('median')

# z-score: how many standard deviations from THIS account's normal?
df_train['cost_zscore'] = (
    (df_train['cost'] - df_train['acct_svc_mean']) /
    (df_train['acct_svc_std'] + 1e-9)
).clip(-10, 10)  # clip to prevent extreme values

# cost_ratio: cost relative to account's p95 (>1 means above 95th percentile for this account)
df_train['cost_ratio_p95'] = (
    df_train['cost'] / (df_train['acct_svc_p95'] + 1e-9)
).clip(0, 100)

# cost_ratio_mean: how many times the normal cost for this account+service
df_train['cost_ratio_mean'] = (
    df_train['cost'] / (df_train['acct_svc_mean'] + 1e-9)
).clip(0, 100)

# Account-level total spend per day (detects account-wide spending spikes)
df_train['date'] = df_train['timestamp'].dt.date
daily_acct_cost = df_train.groupby(['account_id', 'date'])['cost'].sum().reset_index()
daily_acct_cost.columns = ['account_id', 'date', 'daily_acct_total']
df_train = df_train.merge(daily_acct_cost, on=['account_id', 'date'], how='left')

acct_daily_mean = df_train.groupby('account_id')['daily_acct_total'].transform('mean')
acct_daily_std  = df_train.groupby('account_id')['daily_acct_total'].transform('std').fillna(0)
df_train['daily_spend_zscore'] = (
    (df_train['daily_acct_total'] - acct_daily_mean) /
    (acct_daily_std + 1e-9)
).clip(-10, 10)

# ── Cost efficiency: cost per unit of usage ───────────────────────────────────
# Spike in cost WITH no usage increase = strong anomaly signal
df_train['cost_per_unit'] = (
    df_train['cost'] / df_train['usage_amount'].replace(0, np.nan)
).fillna(0).clip(0, df_train['cost'].quantile(0.999) * 10)

# Normalize cost_per_unit per account+service (same multi-account logic)
df_train['cpu_mean'] = df_train.groupby(grp_key)['cost_per_unit'].transform('mean')
df_train['cost_per_unit_ratio'] = (
    df_train['cost_per_unit'] / (df_train['cpu_mean'] + 1e-9)
).clip(0, 100)

# ── Log transforms ────────────────────────────────────────────────────────────
df_train['log_cost']         = np.log1p(df_train['cost'])
df_train['log_usage_amount'] = np.log1p(df_train['usage_amount'].clip(lower=0))

print('✅ Feature engineering complete')

## 4. Build Feature Matrix

In [ ]:
FEATURE_COLS = [
    # ── Account-normalized cost signals (most important) ──
    'cost_zscore',          # How many SDs from THIS account's normal
    'cost_ratio_p95',       # Is this above the 95th pct for this account+service?
    'cost_ratio_mean',      # How many times the typical cost for this account+service
    'daily_spend_zscore',   # Is today's total account spend unusual?
    'cost_per_unit_ratio',  # Cost efficiency vs this account's normal

    # ── Raw cost (log-transformed) ──
    'log_cost',
    'log_usage_amount',
    'normalized_usage',

    # ── Resource utilization ──
    # High cost + low utilization = strong anomaly
    'cpu_utilization',
    'memory_utilization',
    'network_in_mb',
    'network_out_mb',
    'latency_ms',
    'throughput',
    'invocations',
    'duration_ms',
    'error_count',
    'availability_percent',
    'status_check_failed',

    # ── Temporal ──
    'hour',
    'dayofweek',
    'is_weekend',
    'month',

    # ── Service / usage type (frequency encoded — no more broken Other bucket) ──
    'service_freq',
    'usage_type_freq',
]

X = df_train[FEATURE_COLS].fillna(0).replace([np.inf, -np.inf], 0)

print(f'Feature matrix: {X.shape}')
print(f'Any NaN: {X.isnull().any().any()} | Any Inf: {np.isinf(X.values).any()}')

## 5. Train Isolation Forest

Using **RobustScaler** (median + IQR) instead of StandardScaler (mean + std).
StandardScaler gets distorted by outliers — exactly what you're trying to detect.
RobustScaler's median/IQR is not affected by extreme values.

In [ ]:
# ── Scale ─────────────────────────────────────────────────────────────────────
scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X)

print(f'Training on {X_scaled.shape[0]:,} rows × {X_scaled.shape[1]} features...')

# ── Train ─────────────────────────────────────────────────────────────────────
iso = IsolationForest(
    n_estimators=300,       # More trees = more stable
    max_samples=512,        # Sample size per tree
    max_features=0.75,      # 75% of features per tree — adds diversity
    contamination=0.01,     # Expect ~1% anomalies — tune if needed
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_scaled)

# ── Score ─────────────────────────────────────────────────────────────────────
df_train['raw_score']     = iso.decision_function(X_scaled)  # negative = anomalous
df_train['anomaly_label'] = iso.predict(X_scaled)            # -1 = anomaly, 1 = normal
df_train['is_anomaly']    = (df_train['anomaly_label'] == -1).astype(int)

# Normalize to 0–1 where 1 = most anomalous
s_min = df_train['raw_score'].min()
s_max = df_train['raw_score'].max()
df_train['anomaly_score'] = 1 - (df_train['raw_score'] - s_min) / (s_max - s_min + 1e-9)

n_anom = df_train['is_anomaly'].sum()
print(f'\n✅ Done. Anomalies: {n_anom:,} / {len(df_train):,} ({100*n_anom/len(df_train):.2f}%)')

## 6. Validate — Do the Anomalies Make Sense?

In [ ]:
# ── Check 1: Anomalies should have higher cost_zscore ─────────────────────────
print('=== cost_zscore: Normal vs Anomaly ===')
print(df_train.groupby('is_anomaly')['cost_zscore'].describe())
print()

# ── Check 2: Anomaly rate by service ─────────────────────────────────────────
print('=== Anomaly rate by service ===')
svc_summary = (
    df_train.groupby('service')
    .agg(total=('is_anomaly','count'), anomalies=('is_anomaly','sum'))
)
svc_summary['rate'] = svc_summary['anomalies'] / svc_summary['total']
print(svc_summary.sort_values('rate', ascending=False).to_string())
print()

# ── Check 3: Top anomalies should have high cost_zscore or high cost_ratio ────
print('=== Top 15 most anomalous rows ===')
cols_show = ['timestamp','service','usage_type','cost','cost_zscore',
             'cost_ratio_p95','daily_spend_zscore','cpu_utilization',
             'error_count','anomaly_score']
print(df_train[df_train['is_anomaly']==1].nlargest(15,'anomaly_score')[cols_show].to_string())

In [ ]:
# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('AWS Cost Anomaly Detection — Isolation Forest (Fixed)', fontsize=15, fontweight='bold')

normal = df_train[df_train['is_anomaly'] == 0]
anomal = df_train[df_train['is_anomaly'] == 1]

# Plot 1: Anomaly score distribution
# GOOD result: normal = left cluster (0–0.4), anomalies = right tail (0.6–1.0)
ax = axes[0, 0]
normal['anomaly_score'].hist(bins=60, ax=ax, alpha=0.7, color='steelblue', label=f'Normal ({len(normal):,})')
anomal['anomaly_score'].hist(bins=60, ax=ax, alpha=0.8, color='crimson', label=f'Anomaly ({len(anomal):,})')
ax.set_title('Anomaly Score Distribution\n(Normal should be left, Anomalies right)')
ax.set_xlabel('Anomaly Score (0=normal, 1=most anomalous)')
ax.set_ylabel('Count')
ax.axvline(x=0.5, color='black', linestyle='--', alpha=0.5, label='0.5 threshold')
ax.legend()

# Plot 2: cost_zscore vs anomaly_score
# GOOD result: high zscore rows should have high anomaly score
ax = axes[0, 1]
n_sample = normal.sample(min(5000, len(normal)), random_state=42)
ax.scatter(n_sample['cost_zscore'], n_sample['anomaly_score'], alpha=0.2, s=5, color='steelblue', label='Normal')
ax.scatter(anomal['cost_zscore'], anomal['anomaly_score'], alpha=0.6, s=15, color='crimson', label='Anomaly')
ax.set_title('Cost Z-Score vs Anomaly Score\n(High zscore = high score = correct)')
ax.set_xlabel('Cost Z-Score (account-normalized)')
ax.set_ylabel('Anomaly Score')
ax.legend()

# Plot 3: Anomaly rate by service (should all be near 1%)
ax = axes[1, 0]
svc_summary['rate'].sort_values(ascending=True).plot(kind='barh', ax=ax, color='coral')
ax.axvline(x=0.01, color='black', linestyle='--', label='1% baseline')
ax.set_title('Anomaly Rate by Service\n(Should be roughly even, ~1%)')
ax.set_xlabel('Anomaly Rate')
ax.legend()

# Plot 4: Cost distribution of anomalies vs normal — log scale
# GOOD result: anomalies should skew toward HIGHER costs
ax = axes[1, 1]
p99 = df_train['cost'].quantile(0.99)
normal['cost'].clip(upper=p99).hist(bins=50, ax=ax, alpha=0.7, color='steelblue', label='Normal', density=True)
anomal['cost'].clip(upper=p99).hist(bins=50, ax=ax, alpha=0.7, color='crimson', label='Anomaly', density=True)
ax.set_title('Cost Distribution: Normal vs Anomaly\n(Anomalies should skew higher)')
ax.set_xlabel('Cost (USD, clipped at p99)')
ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.savefig('anomaly_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: anomaly_plots.png')

## 7. Save Artifacts

In [ ]:
# ── Model + scaler ────────────────────────────────────────────────────────────
joblib.dump(iso,    'isolation_forest_model.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')

# ── Feature list (exact order matters for inference) ─────────────────────────
with open('feature_columns.json', 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)

# ── Score normalization params ────────────────────────────────────────────────
with open('score_norm_params.json', 'w') as f:
    json.dump({'score_min': float(s_min), 'score_max': float(s_max)}, f, indent=2)

# ── Per account+service baseline stats (needed for inference on new data) ─────
# This is the lookup table your website backend uses to compute
# cost_zscore / cost_ratio_p95 for incoming rows
baseline = (
    df_train.groupby(['account_id', 'service', 'usage_type'])
    .agg(
        mean_cost   = ('cost', 'mean'),
        std_cost    = ('cost', 'std'),
        p95_cost    = ('cost', lambda x: x.quantile(0.95)),
        median_cost = ('cost', 'median'),
        mean_cpu    = ('cost_per_unit', 'mean'),
    )
    .fillna(0)
    .reset_index()
)
baseline.to_csv('account_service_baseline.csv', index=False)

# ── Frequency encoding maps ───────────────────────────────────────────────────
with open('encoding_maps.json', 'w') as f:
    json.dump({'service_freq': svc_freq, 'usage_type_freq': ut_freq}, f, indent=2)

# ── Results CSV ───────────────────────────────────────────────────────────────
out_cols = ['timestamp','account_id','service','usage_type','cost',
            'cost_zscore','cost_ratio_p95','daily_spend_zscore',
            'cpu_utilization','error_count','anomaly_score','is_anomaly']
df_train[out_cols].to_csv('aws_billing_with_anomalies.csv', index=False)

print('✅ Saved all artifacts:')
for f in ['isolation_forest_model.pkl','feature_scaler.pkl','feature_columns.json',
          'score_norm_params.json','account_service_baseline.csv',
          'encoding_maps.json','aws_billing_with_anomalies.csv']:
    print(f'   {f}')

## 8. Download (Colab)

In [ ]:
from google.colab import files
for fname in ['isolation_forest_model.pkl','feature_scaler.pkl','feature_columns.json',
              'score_norm_params.json','account_service_baseline.csv',
              'encoding_maps.json','aws_billing_with_anomalies.csv','anomaly_plots.png']:
    files.download(fname)
    print(f'📥 {fname}')

## 9. Inference Function (paste into your website backend)

Load artifacts once at startup. Call `predict_anomaly()` per incoming billing row.

In [ ]:
import joblib, json, numpy as np, pandas as pd

# ── Load once at startup ──────────────────────────────────────────────────────
MODEL        = joblib.load('isolation_forest_model.pkl')
SCALER       = joblib.load('feature_scaler.pkl')
FEAT_COLS    = json.load(open('feature_columns.json'))
NORM         = json.load(open('score_norm_params.json'))
ENC_MAPS     = json.load(open('encoding_maps.json'))
BASELINE_DF  = pd.read_csv('account_service_baseline.csv')

# Convert baseline to fast lookup dict: (account_id, service, usage_type) -> stats
BASELINE = {
    (r.account_id, r.service, r.usage_type): r
    for _, r in BASELINE_DF.iterrows()
}


def predict_anomaly(row: dict) -> dict:
    """
    Predict anomaly for one AWS billing row.

    row must contain at minimum:
        account_id, service, usage_type, cost, usage_amount, timestamp
        + utilization metrics (cpu_utilization, memory_utilization, etc.)

    Returns:
        is_anomaly (bool), anomaly_score (0–1), explanation (dict)
    """
    key = (str(row.get('account_id','')), row.get('service',''), row.get('usage_type',''))
    b   = BASELINE.get(key)

    # If new account/service not in baseline, use global fallback
    mean_cost = float(b.mean_cost)   if b is not None else 0.0
    std_cost  = float(b.std_cost)    if b is not None else 0.001
    p95_cost  = float(b.p95_cost)    if b is not None else 0.01
    mean_cpu  = float(b.mean_cpu)    if b is not None else 0.001

    cost         = float(row.get('cost', 0))
    usage_amount = float(row.get('usage_amount', 0)) or 1

    ts = pd.to_datetime(row.get('timestamp', pd.Timestamp.now()), utc=True, errors='coerce')

    cost_per_unit       = cost / usage_amount
    cost_per_unit_ratio = cost_per_unit / (mean_cpu + 1e-9)

    feat = {
        'cost_zscore':        float(np.clip((cost - mean_cost) / (std_cost + 1e-9), -10, 10)),
        'cost_ratio_p95':     float(np.clip(cost / (p95_cost + 1e-9), 0, 100)),
        'cost_ratio_mean':    float(np.clip(cost / (mean_cost + 1e-9), 0, 100)),
        'daily_spend_zscore': float(row.get('daily_spend_zscore', 0)),  # pre-compute outside
        'cost_per_unit_ratio':float(np.clip(cost_per_unit_ratio, 0, 100)),
        'log_cost':           float(np.log1p(max(cost, 0))),
        'log_usage_amount':   float(np.log1p(max(usage_amount, 0))),
        'normalized_usage':   float(row.get('normalized_usage', 0)),
        'cpu_utilization':    float(row.get('cpu_utilization', 0)),
        'memory_utilization': float(row.get('memory_utilization', 0)),
        'network_in_mb':      float(row.get('network_in_mb', 0)),
        'network_out_mb':     float(row.get('network_out_mb', 0)),
        'latency_ms':         float(row.get('latency_ms', 0)),
        'throughput':         float(row.get('throughput', 0)),
        'invocations':        float(row.get('invocations', 0)),
        'duration_ms':        float(row.get('duration_ms', 0)),
        'error_count':        float(row.get('error_count', 0)),
        'availability_percent': float(row.get('availability_percent', 100)),
        'status_check_failed':  float(row.get('status_check_failed', 0)),
        'hour':       ts.hour if ts else 0,
        'dayofweek':  ts.dayofweek if ts else 0,
        'is_weekend': int(ts.dayofweek >= 5) if ts else 0,
        'month':      ts.month if ts else 1,
        'service_freq':    ENC_MAPS['service_freq'].get(row.get('service',''), 0),
        'usage_type_freq': ENC_MAPS['usage_type_freq'].get(row.get('usage_type',''), 0),
    }

    X = np.array([[feat[c] for c in FEAT_COLS]])
    X = np.nan_to_num(X, nan=0, posinf=0, neginf=0)
    X_scaled = SCALER.transform(X)

    label     = MODEL.predict(X_scaled)[0]
    raw       = float(MODEL.decision_function(X_scaled)[0])
    score     = float(np.clip(1 - (raw - NORM['score_min']) / (NORM['score_max'] - NORM['score_min'] + 1e-9), 0, 1))

    return {
        'is_anomaly':    label == -1,
        'anomaly_score': round(score, 4),
        'explanation': {
            'cost_zscore':     round(feat['cost_zscore'], 2),
            'cost_ratio_p95':  round(feat['cost_ratio_p95'], 2),
            'cost_ratio_mean': round(feat['cost_ratio_mean'], 2),
            'error_count':     feat['error_count'],
        }
    }


# ── Smoke test ────────────────────────────────────────────────────────────────
test_normal = {
    'account_id': '195000000000', 'service': 'Amazon Elastic Compute Cloud',
    'usage_type': 'EBS:VolumeUsage.gp3', 'cost': 0.00134,
    'usage_amount': 1.0, 'normalized_usage': 0.0,
    'cpu_utilization': 45.0, 'memory_utilization': 60.0,
    'network_in_mb': 100.0, 'network_out_mb': 80.0,
    'latency_ms': 12.0, 'throughput': 50.0,
    'invocations': 0, 'duration_ms': 0.0, 'error_count': 0,
    'availability_percent': 99.9, 'status_check_failed': 0,
    'timestamp': '2026-04-15T10:00:00+00:00',
}
test_anomaly = {
    **test_normal,
    'cost': 0.073,           # near-max cost in dataset
    'cpu_utilization': 2.0,  # near-zero CPU with high cost
    'error_count': 45,
}

print('Normal:', predict_anomaly(test_normal))
print('Anomaly:', predict_anomaly(test_anomaly))

## 10. Tuning Contamination

If anomaly rate feels wrong, adjust `contamination` in Cell 5:

| contamination | Flags as anomaly | Use when |
|---|---|---|
| 0.005 | 0.5% of rows | Too many false alarms |
| 0.01 | 1% of rows | Default (good start) |
| 0.02 | 2% of rows | Missing obvious anomalies |
| 0.05 | 5% of rows | Very noisy data |

**Signs the model is working correctly:**
- Top anomalies have `cost_zscore > 3` or `cost_ratio_p95 > 5`
- Anomaly rate is roughly even across services (not one service at 100%)
- Score histogram: normal = left cluster, anomalies = right tail with clear separation